In [ ]:
"""
select_most_unique_images.py

Finds the most "unique" images in one or more input folders using
OpenCLIP embeddings + ChromaDB.

For each image:
    - Compare its embedding against every other image.
    - Calculate the average distance to all other images.
    - Images with HIGHER average distance are considered more unique.
    - Copy the top N most unique images to the output folder.

Requirements:
    pip install chromadb open-clip-torch torch pillow tqdm pandas

Usage:
    python select_most_unique_images.py
"""
from pathlib import Path
import shutil

import chromadb
import open_clip
import torch
from PIL import Image
from tqdm import tqdm
import pandas as pd
import os


#from pathlib import Path 
#TRAIN_DIR = Path("/Users/boy/Desktop/licenseplate-dataset/train")
#OUTPUT_FOLDER = Path("./OUTPUT")
#os.makedirs(OUTPUT_FOLDER, exist_ok = True)

# ============================================================
# CONFIGURATION
# ============================================================

# You can provide multiple input folders
INPUT_FOLDERS = [
    r"/Users/boy/Desktop/licenseplate-dataset/train"
]

OUTPUT_FOLDER = r"./selected_images"
os.makedirs(OUTPUT_FOLDER, exist_ok = True)
# Number of most unique images to select
TOP_N = 10

# ChromaDB temporary/persistent database
CHROMA_DB_PATH = r"./chroma_db"

# Chroma collection name
COLLECTION_NAME = "image_embeddings"

# CLIP model
MODEL_NAME = "ViT-B-32"
PRETRAINED = "laion2b_s34b_b79k"

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Supported image extensions
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
    ".tif",
    ".tiff",
}


# ============================================================
# FIND IMAGES
# ============================================================

def find_images(input_folders):
    """Find all images recursively from input folders."""

    image_paths = []

    for folder in input_folders:
        folder = Path(folder)

        if not folder.exists():
            print(f"WARNING: Folder does not exist: {folder}")
            continue

        for path in folder.rglob("*"):
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
                image_paths.append(path)

    # Remove duplicates while preserving order
    image_paths = list(dict.fromkeys(image_paths))

    return image_paths


# ============================================================
# LOAD CLIP
# ============================================================

def load_model():
    print(f"Using device: {DEVICE}")
    print(f"Loading CLIP model: {MODEL_NAME}")

    model, _, preprocess = open_clip.create_model_and_transforms(
        MODEL_NAME,
        pretrained=PRETRAINED,
        device=DEVICE,
    )

    model.eval()

    return model, preprocess


# ============================================================
# GENERATE EMBEDDINGS
# ============================================================

def generate_embeddings(image_paths, model, preprocess):
    """Generate normalized CLIP embeddings."""

    embeddings = []
    valid_paths = []

    print("\nGenerating image embeddings...")

    with torch.no_grad():

        for image_path in tqdm(image_paths):

            try:
                image = Image.open(image_path).convert("RGB")
                image_tensor = preprocess(image).unsqueeze(0).to(DEVICE)

                embedding = model.encode_image(image_tensor)

                # Normalize embedding
                embedding = embedding / embedding.norm(
                    dim=-1,
                    keepdim=True
                )

                embedding = embedding.cpu().numpy()[0].tolist()

                embeddings.append(embedding)
                valid_paths.append(image_path)

            except Exception as e:
                print(f"\nSkipping {image_path}")
                print(f"Reason: {e}")

    return valid_paths, embeddings


# ============================================================
# CREATE CHROMA DATABASE
# ============================================================

def create_chroma_collection():
    client = chromadb.PersistentClient(
        path=CHROMA_DB_PATH
    )

    # Delete old collection if it exists
    try:
        client.delete_collection(COLLECTION_NAME)


    except Exception:
        
        pass
    collection = client.create_collection(
        name=COLLECTION_NAME,
        metadata={
            "hnsw:space": "cosine"
        }
    )

    return collection


# ============================================================
# ADD EMBEDDINGS TO CHROMA
# ============================================================

BATCH_SIZE = 500


def add_embeddings_to_chroma(
    collection,
    image_paths,
    embeddings
):
    print("\nAdding embeddings to ChromaDB...")

    total = len(image_paths)

    for start in tqdm(
        range(0, total, BATCH_SIZE),
        desc="Adding to ChromaDB"
    ):

        end = min(start + BATCH_SIZE, total)

        batch_ids = [
            str(i)
            for i in range(start, end)
        ]

        batch_embeddings = embeddings[start:end]

        batch_documents = [
            str(path)
            for path in image_paths[start:end]
        ]

        collection.add(
            ids=batch_ids,
            embeddings=batch_embeddings,
            documents=batch_documents,
        )

# ============================================================
# CALCULATE AVERAGE DISTANCES
# ============================================================

def calculate_average_distances(
    image_paths,
    embeddings,
):
    """
    Calculate each image's exact average cosine distance to all others.

    The embeddings are already L2-normalized, so cosine distance is:
        1 - dot(image_embedding, other_embedding)

    Using the sum of all embeddings avoids querying every Chroma record for
    every image, which can exceed SQLite's SQL variable limit on large datasets.
    """

    total_images = len(image_paths)

    if total_images != len(embeddings):
        raise ValueError(
            "image_paths and embeddings must have the same length"
        )

    print("\nCalculating distances...")

    embedding_tensor = torch.tensor(
        embeddings,
        dtype=torch.float32,
    )

    embedding_sum = embedding_tensor.sum(dim=0)
    self_similarities = (embedding_tensor * embedding_tensor).sum(dim=1)
    similarity_sums = embedding_tensor.matmul(embedding_sum)

    average_similarities = (
        similarity_sums - self_similarities
    ) / (total_images - 1)
    average_distances = 1.0 - average_similarities

    results = []

    for index, average_distance in enumerate(
        tqdm(
            average_distances.tolist(),
            total=total_images,
        )
    ):
        results.append({
            "image": str(image_paths[index]),
            "average_distance": average_distance,
            "num_comparisons": total_images - 1,
        })

    return results


# ============================================================
# SAVE TOP IMAGES
# ============================================================

def save_top_images(results, output_folder, top_n):
    output_folder = Path(output_folder)

    output_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    # Sort highest average distance first
    results = sorted(
        results,
        key=lambda x: x["average_distance"],
        reverse=True,
    )

    selected = results[:top_n]

    print("\n" + "=" * 70)
    print(f"TOP {len(selected)} MOST UNIQUE IMAGES")
    print("=" * 70)

    for rank, item in enumerate(selected, start=1):

        source = Path(item["image"])

        # Add rank to filename
        destination = output_folder / (
            f"{rank:02d}_"
            f"{source.name}"
        )

        shutil.copy2(
            source,
            destination
        )

        print(
            f"{rank:02d}. "
            f"{source.name} "
            f"(average distance: "
            f"{item['average_distance']:.6f})"
        )

    return selected


# ============================================================
# SAVE CSV
# ============================================================

def save_results_csv(results):
    df = pd.DataFrame(results)

    df = df.sort_values(
        "average_distance",
        ascending=False
    )

    csv_path = Path(OUTPUT_FOLDER) / "image_ranking.csv"

    df.to_csv(
        csv_path,
        index=False
    )

    print(f"\nRanking saved to: {csv_path}")


# ============================================================
# MAIN
# ============================================================

def main():

    print("=" * 70)
    print("IMAGE UNIQUENESS SELECTOR")
    print("=" * 70)

    # --------------------------------------------------------
    # Find images
    # --------------------------------------------------------

    image_paths = find_images(INPUT_FOLDERS)

    print(f"\nFound {len(image_paths)} images.")

    if len(image_paths) < 2:
        print("Need at least 2 images.")
        return

    # --------------------------------------------------------
    # Load CLIP
    # --------------------------------------------------------

    model, preprocess = load_model()

    # --------------------------------------------------------
    # Generate embeddings
    # --------------------------------------------------------

    image_paths, embeddings = generate_embeddings(
        image_paths,
        model,
        preprocess,
    )

    if len(image_paths) < 2:
        print("Not enough valid images.")
        return

    # --------------------------------------------------------
    # Create ChromaDB
    # --------------------------------------------------------

    collection = create_chroma_collection()

    # --------------------------------------------------------
    # Store embeddings
    # --------------------------------------------------------

    add_embeddings_to_chroma(
        collection,
        image_paths,
        embeddings,
    )

    # --------------------------------------------------------
    # Calculate average distances
    # --------------------------------------------------------

    results = calculate_average_distances(
        image_paths,
        embeddings,
    )

    # --------------------------------------------------------
    # Save top images
    # --------------------------------------------------------

    selected = save_top_images(
        results,
        OUTPUT_FOLDER,
        TOP_N,
    )

    # --------------------------------------------------------
    # Save ranking
    # --------------------------------------------------------

    save_results_csv(results)

    print("\nDone.")
    print(f"Selected images: {len(selected)}")
    print(f"Output folder: {OUTPUT_FOLDER}")



In [ ]:
main()

IMAGE UNIQUENESS SELECTOR

Found 98798 images.
Using device: cpu
Loading CLIP model: ViT-B-32

Generating image embeddings...


100%|████████████████████▉| 98752/98798 [1:04:23<00:01, 27.44it/s]

In [6]:
d = pd.read_csv("./selected_images/image_ranking.csv")
df = d.copy(deep = True)
df['image'] = df['image'].apply(lambda x: x[46:])
first_1000_unique_images = df[:100]['image'].to_list()
first_1000_unique_images[:3]

In [15]:
from pathlib import Path
import shutil


def select_images(input_folder, selected_images):
    """
    Copy selected images from an input folder to an output folder.

    Args:
        input_folder: Path to the folder containing the images.
        selected_images: List of image paths or filenames to select.

    Returns:
        List of paths to the copied images.
    """
    input_folder = Path(input_folder)
    output_folder = Path("../data/top_n_selected_images")

    if not input_folder.exists():
        raise FileNotFoundError(
            f"Input folder does not exist: {input_folder}"
        )

    output_folder.mkdir(parents=True, exist_ok=True)

    copied_image_paths = []

    for image_path in selected_images:
        image_path = Path(image_path)

        # If only a filename is provided, look for it inside input_folder
        if not image_path.is_absolute():
            image_path = input_folder / image_path

        if not image_path.exists():
            print(f"WARNING: Image does not exist: {image_path}")
            continue

        destination_path = output_folder / image_path.name

        shutil.copy2(image_path, destination_path)

        copied_image_paths.append(destination_path)

    return copied_image_paths

In [9]:
INPUT_FOLDER = r"/Users/boy/Desktop/licenseplate-dataset/train"


In [ ]:
select_images(INPUT_FOLDER, first_1000_unique_images)